# Movie Browser.

The first things to do are to import the necessary libraries and do initial exploration of the data.

<u>Imports

In [79]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import random



# Use pip to install openpyxl 
df = pd.read_excel("movies_cleaned.xlsx")

print(f"Total number of movies: {len(df)}")
df.head()

Total number of movies: 260


,Year,Movie Title,Genre,Runtime (min),IMDb Rating,Top Figure
0,2000,Gladiator,"Action, Drama",155,8.5,Russell Crowe
1,2000,Memento,"Mystery, Thriller",113,8.4,Guy Pearce
2,2000,X‑Men,"Action, Sci‑Fi",104,7.4,Hugh Jackman
3,2000,Cast Away,"Adventure, Drama",143,7.8,Tom Hanks
4,2000,Chicken Run,"Animation, Comedy",84,7.1,Mel Gibson


The Genre column has multiple values in each row so a new column will be created called Genre_List, thay has the values in Genre as a list. 

This will make those values easier to access.

In [80]:
df['Genre_List'] = df['Genre'].str.split(', ')
df.head()

,Year,Movie Title,Genre,Runtime (min),IMDb Rating,Top Figure,Genre_List
0,2000,Gladiator,"Action, Drama",155,8.5,Russell Crowe,"[Action, Drama]"
1,2000,Memento,"Mystery, Thriller",113,8.4,Guy Pearce,"[Mystery, Thriller]"
2,2000,X‑Men,"Action, Sci‑Fi",104,7.4,Hugh Jackman,"[Action, Sci‑Fi]"
3,2000,Cast Away,"Adventure, Drama",143,7.8,Tom Hanks,"[Adventure, Drama]"
4,2000,Chicken Run,"Animation, Comedy",84,7.1,Mel Gibson,"[Animation, Comedy]"



<u>Function to filter movies.</u>

The user will have the ability to filter for movies based off of;
* A specified period of time.
    * This is done by assigning values to the year_start and year_end parameters.
* The genre.
    * This is done by assigning a value to the genre parameter.
* The rating.
    * This is done by assigning a value to the min_rating parameter.
* The length of the movie.
    * This is done by assigning a value to the max_runtime parameter.

If none of those parameters are assigned a value, the function will return the data frame as it is.

In [81]:
def search_movies(year_start=None, year_end=None, genre=None, min_rating=None, max_runtime=None):
    filtered = df.copy()
    
    if year_start:
        filtered = filtered[filtered['Year'] >= year_start]
    if year_end:
        filtered = filtered[filtered['Year'] <= year_end]
    
    if genre:
        filtered = filtered[filtered['Genre_List'].apply(lambda x: genre in x)]
    
    if min_rating:
        filtered = filtered[filtered['IMDb Rating'] >= min_rating]
    
    if max_runtime:
        filtered = filtered[filtered['Runtime (min)'] <= max_runtime]
    
    filtered = filtered.sort_values('IMDb Rating', ascending= False)
    
    return filtered

Test: Action movie btw 2010 to 2020, rating > 8

In [82]:
results = search_movies(
    year_start= 2010,
    year_end= 2020,
    genre= "Action",
    min_rating= 8.0
)

print(f"Found {len(results)} movies:")
results[['Year', 'Movie Title', 'Genre', 'IMDb Rating']].head(10)  # Show top 10

Found 12 movies:


,Year,Movie Title,Genre,IMDb Rating
100,2010,Inception,"Action, Sci‑Fi",8.8
120,2012,The Dark Knight Rises,"Action, Thriller",8.4
180,2018,Avengers: Infinity War,"Action, Adventure",8.4
184,2018,Spider‑Man: Into the Spider‑Verse,"Animation, Action",8.4
191,2019,Avengers: Endgame,"Action, Adventure",8.4
150,2015,Mad Max: Fury Road,"Action, Adventure",8.1
173,2017,Logan,"Action, Drama",8.1
198,2019,Ford v Ferrari,"Action, Biography",8.1
122,2012,The Avengers,"Action, Sci‑Fi",8
145,2014,Guardians of the Galaxy,"Action, Adventure",8


# Insight board

<u>Top 10 rated movies</u>

In [83]:
df["IMDb Rating"].unique()


array([8.5, 8.4, 7.4, 7.8, 7.1, 7.9, 8.3, 8.2, 7.3, 7, 8.8, 8.6, 7.7, 8.1,
       8, 8.7, 7.6, 7.5, 9, 7.2, 6.9, 6.5, '—'], dtype=object)

In [84]:
df["IMDb Rating"] = np.where(df["IMDb Rating"] == "—", np.nan, df["IMDb Rating"])
df["IMDb Rating"].unique()


array([8.5, 8.4, 7.4, 7.8, 7.1, 7.9, 8.3, 8.2, 7.3, 7, 8.8, 8.6, 7.7, 8.1,
       8, 8.7, 7.6, 7.5, 9, 7.2, 6.9, 6.5, nan], dtype=object)

In [ ]:
def top_10_ratings():
    df["IMDb Rating"] = df["IMDb Rating"].astype(float)
    top_movies = df.nlargest(10, 'IMDb Rating')

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data= top_movies, 
        x= 'IMDb Rating', 
        y= 'Movie Title', 
        palette= 'viridis'
    )
    plt.title('Top 10 Highest Rated Movies')
    plt.xlabel('IMDb Rating')
    plt.ylabel('Movie Title')
    plt.show()

<u>Genre breakdowm</u>

In [86]:
def top_10_genre():
    genres_exploded = df['Genre'].str.split(', ').explode()

    genre_counts = genres_exploded.value_counts().reset_index()
    genre_counts.columns = ['Genre', 'Count'] 

    # top_genres = genre_counts.head()

    sns.barplot(
        data=top_genres, 
        y='Genre', 
        x='Count', 
        palette='viridis'
    )
    plt.title('Top 10 Genres')
    plt.xlabel('Number of Movies')
    plt.ylabel('Genre')
    plt.show()

<u>Rating Distribution</u>

In [87]:
def top_10_rating():
    sns.histplot(
        df['IMDb Rating'], 
        bins= 20, 
        kde= True, 
        color= 'skyblue'
    )
    plt.title('Distribution of Ratings')
    plt.xlabel('IMDb Rating')
    plt.ylabel('Number of Movies')
    plt.axvline(df['IMDb Rating'].mean(), color='red', linestyle='--', label=f'Average Rating: {df["IMDb Rating"].mean():.2f}')
    plt.legend()
    plt.show()

# Movie recommeder

Recommends movies based off of genre and rating.

In [88]:

def recommend_movies(movie_title, top_n=5):
   # Check if the movie exists
    movie_row = df[df['Movie Title'].str.lower() == movie_title.lower()]
    
    if movie_row.empty:
        return (f"Sorry, '{movie_title}' does not exist")
    
    favorite_genre = movie_row['Genre_List'].iloc[0]  
    
    fav_genre = favorite_genre[0]  #
    
    recommendations = search_movies(
        genre= fav_genre,
        min_rating= 7.0  
    )
    
    recommendations = recommendations[recommendations['Movie Title'].str.lower() != movie_title.lower()]
    
    return recommendations[['Movie Title', 'Year', 'Genre', 'IMDb Rating']].head(top_n)

# Random recommedation

In [92]:
def surprise_me():
    df["IMDb Rating"] = df["IMDb Rating"].astype(float)
    top_movies = df.nlargest(100, 'IMDb Rating')
    random_movie = top_movies.sample(1).iloc[0]
    return f"How about '{random_movie['Movie Title']}' ({random_movie['Year']}) - Rating: {random_movie['IMDb Rating']}"

# Test it
print(surprise_me())

How about 'Toy Story 3' (2010) - Rating: 8.3
